# Lab 4B — Genie Inside an Agent

In [Lab 4A](#) you curated a Genie Agent and measured it: **0% → 100%**.

Here you make it **one tool among several**. The agent decides *when* a question
needs data, asks Genie in its own words, and combines the answer with the policy
documents from [Session 3](#).

Along the way this lab answers a question Session 3 left open: **when does curation
get inherited by a new consumer, and when does it not?**

| Cell group | What you do |
|---|---|
| 0–2 | Connect, find your Genie Agent, ask it one question over the API. |
| 3 | **Prove the Lab 4A curation is inherited here.** |
| 4–5 | Wrap Genie as a tool beside the policy retriever. |
| 6 | Ask something that needs both. |
| 7 | Where curation lives decides whether it travels. |

**Compute:** Serverless. **Prerequisite:** [Lab 4A](#) — curated, with instructions saved.

In [0]:
%pip install -q openai databricks-ai-search
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## 1. Find your Genie Agent

Spaces **are** scriptable, even though benchmarks and the Knowledge Store are not.
This cell finds yours by title rather than making you paste an id.

In [0]:
import requests
from databricks.sdk import WorkspaceClient

SPACE_TITLE = "Retail Customer Orders"

w = WorkspaceClient()
HOST = w.config.host
TOKEN = w.config.authenticate()["Authorization"].removeprefix("Bearer ")
HEAD = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}


def api(method: str, path: str, **kw):
    r = requests.request(method, f"{HOST}{path}", headers=HEAD, timeout=120, **kw)
    r.raise_for_status()
    return r.json() if r.text else {}


spaces = api("GET", "/api/2.0/genie/spaces").get("spaces", [])
match = [s for s in spaces if s.get("title") == SPACE_TITLE]
if not match:
    raise RuntimeError(
        f"No Genie Agent titled {SPACE_TITLE!r}. Available: {[s.get('title') for s in spaces]}")

SPACE_ID = match[0]["space_id"]
print(f"  agent    : {match[0]['title']}")
print(f"  space_id : {SPACE_ID}")

  agent    : Retail Customer Orders
  space_id : 01f1bc6b0a4f1c6d8269cc9c1ec2af08


## 2. Ask Genie over the API

Genie returns an **answer** and **the SQL it ran**. Both matter, and the SQL is the
one you keep — it is the audit trail, and the thing to read when an answer surprises
you.

The API is a conversation: start it, then poll until the message completes.

> ⚠️ **This lab calls the REST API directly rather than `WorkspaceClient.genie`.**
> The SDK bundled with serverless compute is not always current — the first version of
> this notebook failed with
> `AttributeError: 'WorkspaceClient' object has no attribute 'genie'`, even though the
> same code worked locally on a newer SDK. REST does not have that problem, and it
> shows you what the SDK would have been doing anyway.

In [0]:
import time, json, warnings

warnings.filterwarnings("ignore", message=".*serializer warnings.*")

TERMINAL = {"COMPLETED", "FAILED", "QUERY_RESULT_EXPIRED", "CANCELLED"}
BASE = f"/api/2.0/genie/spaces/{SPACE_ID}"


def ask_genie(question: str, timeout: int = 180) -> dict:
    """Start a conversation, poll to completion, return text + SQL + rows."""
    conv = api("POST", f"{BASE}/start-conversation", json={"content": question})
    cid = conv["conversation_id"]
    mid = conv["message_id"]
    msg_path = f"{BASE}/conversations/{cid}/messages/{mid}"

    for _ in range(timeout):
        m = api("GET", msg_path)
        if m.get("status") in TERMINAL:
            break
        time.sleep(1)

    out = {"question": question, "status": m.get("status"),
           "text": None, "sql": None, "rows": None}

    for att in (m.get("attachments") or []):
        if att.get("text", {}).get("content"):
            out["text"] = att["text"]["content"]
        q = att.get("query")
        if q:
            out["sql"] = q.get("query")
            try:
                res = api("GET", f"{msg_path}/attachments/{att['attachment_id']}/query-result")
                sr = res.get("statement_response", {})
                cols = [c["name"] for c in sr.get("manifest", {}).get("schema", {}).get("columns", [])]
                out["rows"] = [dict(zip(cols, row))
                               for row in (sr.get("result", {}).get("data_array") or [])]
            except Exception as e:
                out["rows"] = [{"error": str(e)[:160]}]
    return out


r = ask_genie("Which region had the biggest drop in revenue from August to September 2026?")
print(f"  status: {r['status']}\n")
print(f"  {r['text']}\n")
print("  rows:", json.dumps(r["rows"], default=str)[:300])

  status: COMPLETED

  The region with the biggest revenue drop from August to September 2026 was **Nordics**. **Nordics** fell from **22020.00** in August 2026 to **4678.00** in September 2026, a drop of **17342.00**.

  rows: [{"region": "Nordics", "august_revenue": "22020.00", "september_revenue": "4678.00", "revenue_drop": "17342.00"}]


## 3. Is the Lab 4A curation inherited here?

This is the cell to slow down for.

In Lab 4A you fixed *"Who are our best customers?"* in the **Knowledge Store** — one
SQL Expression plus five lines of General Instructions. You did that in the console.

**This notebook is a completely different consumer.** Different process, different
client, no knowledge of what you typed into that UI.

Ask it the same question and see what happens.

In [0]:
r = ask_genie("Who are our best customers?")

print(f"  {r['text']}\n")
print("  SQL Genie ran:\n")
print("   ", (r["sql"] or "").replace("\n", "\n    "))
print(f"\n  rows returned: {len(r['rows'] or [])}")

  Would you like to see the best customers by number of orders or average order value instead?

  SQL Genie ran:

    SELECT
      customers.customer_id,
      customers.name,
      SUM(orders.revenue) AS lifetime_revenue
    FROM agents_labs.retail.orders AS orders
    INNER JOIN agents_labs.retail.customers AS customers
      ON orders.customer_id = customers.customer_id
    GROUP BY customers.customer_id, customers.name
    ORDER BY lifetime_revenue DESC, customers.customer_id ASC

  rows returned: 8


### Check the curation against the SQL above

Lab 4A added exactly three things. Test for those, and **only** those.

In [0]:
sql = (r["sql"] or "").lower()
n = len(r["rows"] or [])

# Test the three things you actually curated in Lab 4A -- and only those.
checks = [
    ("uses SUM(orders.revenue)  [the snippet]", "sum(orders.revenue)" in sql.replace(" ", ""), True),
    ("filters to 'delivered'    [fault 1]",     "delivered" in sql,                            False),
    ("groups by tier            [fault 2]",     "tier" in sql,                                 False),
]

print(f"  {'check':44} {'observed':>9} {'wanted':>7}")
for label, observed, want in checks:
    print(f"  {label:44} {str(observed):>9} {str(want):>7}   "
          f"{'OK' if observed == want else 'FAIL'}")

print(f"\n  rows returned: {n}")
print("\n  You changed nothing in this notebook. All three curation rules travelled")
print("  with the agent, because that is where they live.")

  check                                         observed  wanted
  uses SUM(orders.revenue)  [the snippet]           True    True   OK
  filters to 'delivered'    [fault 1]              False   False   OK
  groups by tier            [fault 2]              False   False   OK

  rows returned: 8

  You changed nothing in this notebook. All three curation rules travelled
  with the agent, because that is where they live.


### A note on the row count

Genie returned **8 rows**, not 5 — and that is correct. Your Lab 4A instruction was
*"when the user asks for a top N, return exactly N rows"*, and *"Who are our best
customers?"* does not ask for a top N. The ground-truth SQL had `LIMIT 5` because
**you** decided five was enough, not because the question said so.

> ⚠️ **The first version of this cell checked for exactly 5 rows and reported FAIL.**
> The check was wrong, not the agent — the same mistake as
> [Lab 7B](#)'s refusal detector and [Lab 6A](#)'s judge. Three times now, in this
> course, the measuring instrument has been the broken part.
>
> **Before you trust a check, feed it a case you have read yourself and confirm it
> agrees with you.**

It also means your benchmark's ground truth encodes a choice the question left open.
That is fine — but write it down, because the next person will read `LIMIT 5` as a
rule rather than a preference.

## 4. Wrap Genie as a tool

Now Genie stops being the whole system and becomes one capability.

The tool **description** is what decides when the agent reaches for data rather than
documents — so it is worth as much care as the Knowledge Store.

In [0]:
from openai import OpenAI
from databricks.ai_search.client import VectorSearchClient

_tok = w.config.authenticate()["Authorization"].removeprefix("Bearer ")
client = OpenAI(api_key=_tok, base_url=f"{w.config.host}/serving-endpoints")
MODEL = "databricks-claude-sonnet-5"

CATALOG, SCHEMA = "agents_labs", "retail"
_ix = VectorSearchClient(workspace_url=w.config.host, personal_access_token=_tok,
                         disable_notice=True).get_index(
    endpoint_name="agents-labs-vs", index_name=f"{CATALOG}.{SCHEMA}.support_chunks_idx")


def search_policy(query: str, k: int = 3):
    res = _ix.similarity_search(query_text=query,
                                columns=["doc_id", "title", "chunk"],
                                filters={"audience": "customer"}, num_results=k)
    return [{"doc_id": x[0], "title": x[1], "text": x[2]}
            for x in (res.get("result", {}).get("data_array", []) or [])]


TOOLS = [
    {"type": "function", "function": {
        "name": "query_business_data",
        "description": "Ask a natural-language question about orders, customers, revenue, "
                       "regions or loyalty tiers. Returns an answer plus the SQL that was "
                       "run. Use this for anything requiring actual numbers from the "
                       "warehouse. Do NOT use it for written policy.",
        "parameters": {"type": "object",
                       "properties": {"question": {"type": "string"}},
                       "required": ["question"]}}},
    {"type": "function", "function": {
        "name": "search_policy",
        "description": "Search the written customer-facing policy documents: delivery "
                       "timescales, returns, warranty, billing. Use this for rules, not "
                       "numbers.",
        "parameters": {"type": "object",
                       "properties": {"query": {"type": "string"}},
                       "required": ["query"]}}},
]

IMPL = {
    "query_business_data": lambda a: {k: v for k, v in ask_genie(a["question"]).items()
                                      if k in ("text", "sql", "rows")},
    "search_policy": lambda a: search_policy(a["query"]),
}

print("  tools:", [t["function"]["name"] for t in TOOLS])

  tools: ['query_business_data', 'search_policy']


## 5. The agent loop

In [0]:
SYSTEM = """You are a business analyst for an office furniture retailer.

You have two sources:
  * query_business_data - live numbers from the governed warehouse, via Genie
  * search_policy       - the company's written policies

Use data for "what happened" and policy for "what are the rules". Many questions
need both. Quote the figures you were given rather than estimating, and cite the
policy document id like [DOC-003]. If Genie returns SQL, trust its numbers over
your own arithmetic."""


def _text(m):
    c = m.content
    if c is None: return ""
    if isinstance(c, str): return c
    return "\n".join(b.get("text", "") for b in c
                     if isinstance(b, dict) and b.get("type") == "text").strip()


def run(question: str, max_steps: int = 6) -> str:
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": question}]
    for step in range(1, max_steps + 1):
        msg = client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOLS, max_tokens=1100
        ).choices[0].message
        messages.append(msg.model_dump(exclude_none=True))
        if not msg.tool_calls:
            return _text(msg)
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments or "{}")
            shown = args.get("question") or args.get("query", "")
            print(f"  step {step}: {tc.function.name}({shown[:70]!r})")
            out = IMPL[tc.function.name](args)
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": json.dumps(out, default=str)[:7000]})
    return "(no answer within max_steps)"


print("  agent ready")

  agent ready


## 6. A question that needs both sources

> *"Nordics revenue dropped last month. What caused it, and is there anything in our
> policies that might explain a big customer going quiet?"*

**What to watch:** the agent phrases its own questions for Genie. They will not be
the words you typed, and they are worth reading — a badly phrased one is the same
failure as [Lab 7A](#)'s starved worker.

In [0]:
print(run(
    "Nordics revenue dropped last month. What caused it, and is there anything in our "
    "policies that might explain a big customer going quiet?"
))

  step 1: query_business_data('What caused the drop in Nordics revenue last month compared to the pre')
  step 1: search_policy('customer inactivity churn returns delays that could cause a customer t')
[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Here's what I found, plus an important caveat:

**On the "drop" itself — the data doesn't actually show one**

I pulled last month vs. the prior month for Nordics customers/categories, including returns. The result:

| Customer | Category | Prev month rev | Last month rev | Change | Returns |
|---|---|---|---|---|---|
| Nordic Office Group | Seating | €0.00 | €16,500.00 | **+16,500** | none |
| Helsinki Works | Desks | €0.00 | €3,240.00 | **+3,240** | none |
| Fjord Interiors | Lighting | €0.00 | €2,280.00 | **+2,280** | none |

All three Nordics customer/category combination

## 7. Where curation lives decides whether it travels

Two boundaries in this course, both enforced by "the right thing happens", and only
one of them survives a new consumer.

| | Lab 3A — the `audience` filter | Lab 4A — the Knowledge Store |
|---|---|---|
| Where it lives | in the **calling code** (`filters={...}`) | in the **Genie Agent** |
| New consumer | [Lab 7B](#) Agent Bricks — **no filter control, discloses `DOC-006`** | this notebook — **inherits it, cell 3** |
| Why | every caller must remember | the agent carries it |

> 🚨 **This is the same lesson from both directions.** A rule enforced by the caller
> is a convention; a rule that lives with the resource is a control.
>
> You did not write one line of curation in this notebook, and you got all of it. In
> Lab 7B a no-code tool pointed at the Lab 3A index got none of the filtering, and
> leaked an internal document.
>
> When you design an agent system, ask of every rule: **if someone builds a second
> consumer tomorrow, do they inherit this or re-implement it?**

### Try this before you move on

1. Read the SQL Genie returned in cell 6's tool calls. Would you sign it off?
2. Change `query_business_data`'s **description** to something vague — *"asks about
   stuff"* — and re-run cell 6. Does the agent still choose it correctly?
3. Ask something only the agent-only documents answer, e.g. *"what is the refund
   approval limit?"*. Both tools are filtered; watch it refuse.